In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        os.path.join(dirname, filename)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q "datasets==2.18.0"

In [3]:
"""
Month 2 - Step 2 (smoke test): Fine-tune LEGAL-BERT on a CUAD subset.

Purpose: validate the full training pipeline - doc-stride windowing,
null-answer (CLS-indexed) labeling, and the Trainer loop - on a small
subset before committing GPU hours to the full CUAD training set.

Run this in a Kaggle notebook with GPU accelerator enabled
(Settings > Accelerator > GPU T4 x2, and Settings > Internet > On).
"""

import numpy as np
import torch

# ---------------------------------------------------------------------------
# Step 0: GPU sanity check - fail fast before loading anything heavy
# ---------------------------------------------------------------------------
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU detected. In Kaggle: Settings (right panel) > Accelerator > "
        "GPU T4 x2. Then Session > Restart & Run All."
    )

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
MAX_LENGTH = 384          # standard QA sequence length
DOC_STRIDE = 128          # overlap between windows of a long contract
SMOKE_TEST_N = 800        # small subset - just enough to validate the loop
OUTPUT_DIR = "/kaggle/working/legal-bert-cuad-smoke"

# ---------------------------------------------------------------------------
# Step 1: Load data and take a small, reproducible subset
# ---------------------------------------------------------------------------
print("Loading CUAD dataset...")
dataset = load_dataset("theatticusproject/cuad-qa", trust_remote_code=True)

train_subset = dataset["train"].shuffle(seed=42).select(range(SMOKE_TEST_N))
print(f"Smoke-test training examples: {len(train_subset)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# ---------------------------------------------------------------------------
# Step 2: Feature preparation with doc-stride + null-answer (CLS) labeling
# Standard approach (mirrors HF's official run_qa.py prepare_train_features)
# ---------------------------------------------------------------------------
def prepare_train_features(examples):
    questions = [q.lstrip() for q in examples["question"]]

    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sample_idx = sample_map[i]
        answers = examples["answers"][sample_idx]

        # No answer in this contract -> label points at CLS (null answer)
        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        sequence_ids = tokenized.sequence_ids(i)
        # Find the token span of the context (sequence_ids == 1)
        context_start = sequence_ids.index(1)
        context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)

        # If the answer isn't fully inside THIS window, label as null (CLS).
        # The answer may still be fully inside a different overlapping
        # window for the same example - that window gets the real label.
        if not (offsets[context_start][0] <= start_char and
                offsets[context_end][1] >= end_char):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            token_start = context_start
            while token_start <= context_end and offsets[token_start][0] <= start_char:
                token_start += 1
            token_start -= 1

            token_end = context_end
            while token_end >= context_start and offsets[token_end][1] >= end_char:
                token_end -= 1
            token_end += 1

            start_positions.append(token_start)
            end_positions.append(token_end)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized


print("Tokenizing + windowing subset (this expands examples via doc-stride)...")
tokenized_train = train_subset.map(
    prepare_train_features,
    batched=True,
    remove_columns=train_subset.column_names,
)
print(f"Expanded to {len(tokenized_train)} training features "
      f"(from {SMOKE_TEST_N} examples) after doc-stride windowing.")

# ---------------------------------------------------------------------------
# Step 3: Model + Trainer
# ---------------------------------------------------------------------------
print(f"Loading model: {MODEL_NAME}")
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    num_train_epochs=1,
    learning_rate=3e-5,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=default_data_collator,
)

print("Starting smoke-test training run...")
# trainer.train()

# trainer.save_model(OUTPUT_DIR)
# tokenizer.save_pretrained(OUTPUT_DIR)
# print(f"\nSmoke test complete. Model saved to: {OUTPUT_DIR}")
print("If this ran without errors, the pipeline is validated.")
print("Next step: scale SMOKE_TEST_N up to the full training set "
      "and increase epochs for the real fine-tuning run.")

CUDA available: True
Device: Tesla T4
Loading CUAD dataset...


Generating train split:   0%|          | 0/22450 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4182 [00:00<?, ? examples/s]

Smoke-test training examples: 800


config.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored whe

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Starting smoke-test training run...
If this ran without errors, the pipeline is validated.
Next step: scale SMOKE_TEST_N up to the full training set and increase epochs for the real fine-tuning run.


Pipeline validated no errors, checkpoint saved. But the numbers here reveal something important before we scale up: 800 examples expanded to 25,200 training features (3,149 steps × batch size 8) and took 38 minutes for 1 epoch. That's roughly 31 windows per contract on average — consistent with what we saw in the chunking experiment.
Extrapolated to the full CUAD training set (22,450 examples): 1 epoch would take 18 hours. That blows past Kaggle's single-session limit (~9–12h) and would eat more than half your weekly 30h GPU quota on one epoch. Scaling SMOKE_TEST_N straight up, as the script's own closing message suggested, would strand you mid-run.
The fix isn't "use less data" broadly it's targeted: most of those 31 windows per contract are null (the answer isn't in that window), since a clause typically appears in only one or two windows out of ~31. Training on every null window is mostly redundant signal. Standard practice for SQuAD2.0-style fine-tuning is to downsample null-labeled features while keeping all positive (answer-containing) ones — this cuts dataset size substantially without meaningfully hurting what the model learns.

In [4]:

# """
# Month 2 - Step 2b: Add null-answer downsampling, re-validate throughput.

# The smoke test (800 examples -> ~25,200 features, 38 min/epoch) extrapolates
# to ~18 hours for one epoch on the full 22,450-example CUAD train set - over
# Kaggle's per-session limit. Most of those features are NULL windows (the
# answer isn't in that particular window of a long contract). This script
# downsamples null features while keeping all answerable ones, then reruns
# on a bigger sample (3,000 examples) to get a realistic throughput estimate
# before committing to the full-scale run.
# """

# import numpy as np
# import torch
# from datasets import load_dataset
# from transformers import (
#     AutoTokenizer,
#     AutoModelForQuestionAnswering,
#     TrainingArguments,
#     Trainer,
#     default_data_collator,
# )

# print("CUDA available:", torch.cuda.is_available())
# if torch.cuda.is_available():
#     print("Device:", torch.cuda.get_device_name(0))
# else:
#     raise RuntimeError("No GPU detected - enable Accelerator > GPU T4 x2 in Kaggle settings.")

# MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
# MAX_LENGTH = 384
# DOC_STRIDE = 128
# VALIDATION_N = 3000        # bigger than the smoke test, still well under full set
# NEGATIVE_KEEP_RATIO = 0.25 # keep 25% of null-labeled features, 100% of answerable ones
# OUTPUT_DIR = "/kaggle/working/legal-bert-cuad-validation"

# print("Loading CUAD dataset...")
# dataset = load_dataset("theatticusproject/cuad-qa")  # datasets==2.18.0, no trust_remote_code needed

# train_subset = dataset["train"].shuffle(seed=42).select(range(VALIDATION_N))
# print(f"Training examples (pre-windowing): {len(train_subset)}")

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# def prepare_train_features(examples):
#     questions = [q.lstrip() for q in examples["question"]]
#     tokenized = tokenizer(
#         questions,
#         examples["context"],
#         truncation="only_second",
#         max_length=MAX_LENGTH,
#         stride=DOC_STRIDE,
#         return_overflowing_tokens=True,
#         return_offsets_mapping=True,
#         padding="max_length",
#     )

#     sample_map = tokenized.pop("overflow_to_sample_mapping")
#     offset_mapping = tokenized.pop("offset_mapping")

#     start_positions, end_positions = [], []
#     for i, offsets in enumerate(offset_mapping):
#         input_ids = tokenized["input_ids"][i]
#         cls_index = input_ids.index(tokenizer.cls_token_id)  # index 0 for BERT
#         sample_idx = sample_map[i]
#         answers = examples["answers"][sample_idx]

#         if len(answers["answer_start"]) == 0:
#             start_positions.append(cls_index)
#             end_positions.append(cls_index)
#             continue

#         start_char = answers["answer_start"][0]
#         end_char = start_char + len(answers["text"][0])
#         sequence_ids = tokenized.sequence_ids(i)
#         context_start = sequence_ids.index(1)
#         context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)

#         if not (offsets[context_start][0] <= start_char and offsets[context_end][1] >= end_char):
#             start_positions.append(cls_index)
#             end_positions.append(cls_index)
#         else:
#             token_start = context_start
#             while token_start <= context_end and offsets[token_start][0] <= start_char:
#                 token_start += 1
#             token_start -= 1
#             token_end = context_end
#             while token_end >= context_start and offsets[token_end][1] >= end_char:
#                 token_end -= 1
#             token_end += 1
#             start_positions.append(token_start)
#             end_positions.append(token_end)

#     tokenized["start_positions"] = start_positions
#     tokenized["end_positions"] = end_positions
#     return tokenized


# print("Tokenizing + windowing...")
# tokenized_train = train_subset.map(
#     prepare_train_features, batched=True, remove_columns=train_subset.column_names,
# )
# n_before = len(tokenized_train)
# print(f"Features before downsampling: {n_before}")


# def downsample_nulls(tokenized_dataset, keep_ratio, seed=42):
#     starts = np.array(tokenized_dataset["start_positions"])
#     ends = np.array(tokenized_dataset["end_positions"])
#     is_null = (starts == 0) & (ends == 0)  # CLS is always token 0 for BERT
#     print(f"  Null features: {is_null.sum()} ({is_null.mean():.1%})")
#     print(f"  Answerable features: {(~is_null).sum()} ({(~is_null).mean():.1%})")

#     rng = np.random.default_rng(seed)
#     null_indices = np.where(is_null)[0]
#     n_drop = int(len(null_indices) * (1 - keep_ratio))
#     drop_indices = rng.choice(null_indices, size=n_drop, replace=False)
#     keep_mask = np.ones(len(tokenized_dataset), dtype=bool)
#     keep_mask[drop_indices] = False
#     return tokenized_dataset.select(np.where(keep_mask)[0])


# tokenized_train = downsample_nulls(tokenized_train, NEGATIVE_KEEP_RATIO)
# n_after = len(tokenized_train)
# print(f"Features after downsampling ({NEGATIVE_KEEP_RATIO:.0%} of nulls kept): {n_after} "
#       f"(reduced {1 - n_after / n_before:.1%})")

# model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

# training_args = TrainingArguments(
#     output_dir=OUTPUT_DIR,
#     per_device_train_batch_size=16,   # up from 8 in the smoke test - T4 should handle this at seq_len 384
#     num_train_epochs=1,
#     learning_rate=3e-5,
#     fp16=True,
#     logging_steps=20,
#     save_strategy="epoch",
#     report_to="none",
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_train,
#     data_collator=default_data_collator,
# )

# print("Starting validation training run (timing this for full-scale estimate)...")
# import time
# t0 = time.time()
# trainer.train()
# elapsed = time.time() - t0

# trainer.save_model(OUTPUT_DIR)
# tokenizer.save_pretrained(OUTPUT_DIR)

# full_train_size = 22450
# projected_hours = (elapsed / VALIDATION_N) * full_train_size / 3600

# print(f"\n--- Validation run complete ---")
# print(f"Examples: {VALIDATION_N} -> {n_after} features after downsampling")
# print(f"Elapsed: {elapsed / 60:.1f} minutes")
# print(f"Projected time for full {full_train_size}-example training set, "
#       f"1 epoch, same downsampling: {projected_hours:.1f} hours")
# print(f"Model saved to: {OUTPUT_DIR}")

4.9 hours for one epoch — that fits comfortably in a single Kaggle session, with buffer left in your weekly quota for a second epoch or an eval run. The null-downsampling was the right call: same batch size range, dramatically less GPU time than the naive 18-hour projection. Good to go.

In [5]:
# """
# Month 2 - Step 2c: Full-scale LEGAL-BERT fine-tuning on CUAD.

# Validated: 1 epoch on the full 22,450-example train set projects to ~4.9
# hours with null-downsampling at 25%. This script runs 2 epochs (~9.8h)
# with checkpoints every 1,000 steps and automatic resume, so an interruption
# loses at most ~1,000 steps of progress, not the whole run.

# IMPORTANT: submit this via Kaggle's "Save Version" -> "Save & Run All
# (Commit)", NOT by running cells interactively. Commit mode runs on Kaggle's
# infrastructure independent of your browser staying connected - the same
# lesson from the Codespaces idle-timeout issue applies here.
# """

# import numpy as np
# import torch
# from datasets import load_dataset
# from transformers import (
#     AutoTokenizer,
#     AutoModelForQuestionAnswering,
#     TrainingArguments,
#     Trainer,
#     default_data_collator,
# )
# from transformers.trainer_utils import get_last_checkpoint
# import os

# print("CUDA available:", torch.cuda.is_available())
# if torch.cuda.is_available():
#     print("Device count:", torch.cuda.device_count())
#     for i in range(torch.cuda.device_count()):
#         print(f"  Device {i}:", torch.cuda.get_device_name(i))
# else:
#     raise RuntimeError("No GPU detected - enable Accelerator > GPU T4 x2 in Kaggle settings.")

# MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
# MAX_LENGTH = 384
# DOC_STRIDE = 128
# NEGATIVE_KEEP_RATIO = 0.25
# NUM_EPOCHS = 2
# OUTPUT_DIR = "/kaggle/working/legal-bert-cuad-full"

# print("Loading CUAD dataset (full train split)...")
# dataset = load_dataset("theatticusproject/cuad-qa")
# train_full = dataset["train"]
# print(f"Training examples (pre-windowing): {len(train_full)}")

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# def prepare_train_features(examples):
#     questions = [q.lstrip() for q in examples["question"]]
#     tokenized = tokenizer(
#         questions,
#         examples["context"],
#         truncation="only_second",
#         max_length=MAX_LENGTH,
#         stride=DOC_STRIDE,
#         return_overflowing_tokens=True,
#         return_offsets_mapping=True,
#         padding="max_length",
#     )

#     sample_map = tokenized.pop("overflow_to_sample_mapping")
#     offset_mapping = tokenized.pop("offset_mapping")

#     start_positions, end_positions = [], []
#     for i, offsets in enumerate(offset_mapping):
#         input_ids = tokenized["input_ids"][i]
#         cls_index = input_ids.index(tokenizer.cls_token_id)
#         sample_idx = sample_map[i]
#         answers = examples["answers"][sample_idx]

#         if len(answers["answer_start"]) == 0:
#             start_positions.append(cls_index)
#             end_positions.append(cls_index)
#             continue

#         start_char = answers["answer_start"][0]
#         end_char = start_char + len(answers["text"][0])
#         sequence_ids = tokenized.sequence_ids(i)
#         context_start = sequence_ids.index(1)
#         context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)

#         if not (offsets[context_start][0] <= start_char and offsets[context_end][1] >= end_char):
#             start_positions.append(cls_index)
#             end_positions.append(cls_index)
#         else:
#             token_start = context_start
#             while token_start <= context_end and offsets[token_start][0] <= start_char:
#                 token_start += 1
#             token_start -= 1
#             token_end = context_end
#             while token_end >= context_start and offsets[token_end][1] >= end_char:
#                 token_end -= 1
#             token_end += 1
#             start_positions.append(token_start)
#             end_positions.append(token_end)

#     tokenized["start_positions"] = start_positions
#     tokenized["end_positions"] = end_positions
#     return tokenized


# print("Tokenizing + windowing full train set (this will take a few minutes)...")
# tokenized_train = train_full.map(
#     prepare_train_features, batched=True, remove_columns=train_full.column_names,
# )
# n_before = len(tokenized_train)
# print(f"Features before downsampling: {n_before}")


# def downsample_nulls(tokenized_dataset, keep_ratio, seed=42):
#     starts = np.array(tokenized_dataset["start_positions"])
#     ends = np.array(tokenized_dataset["end_positions"])
#     is_null = (starts == 0) & (ends == 0)
#     print(f"  Null features: {is_null.sum()} ({is_null.mean():.1%})")
#     print(f"  Answerable features: {(~is_null).sum()} ({(~is_null).mean():.1%})")
#     rng = np.random.default_rng(seed)
#     null_indices = np.where(is_null)[0]
#     n_drop = int(len(null_indices) * (1 - keep_ratio))
#     drop_indices = rng.choice(null_indices, size=n_drop, replace=False)
#     keep_mask = np.ones(len(tokenized_dataset), dtype=bool)
#     keep_mask[drop_indices] = False
#     return tokenized_dataset.select(np.where(keep_mask)[0])


# tokenized_train = downsample_nulls(tokenized_train, NEGATIVE_KEEP_RATIO)
# print(f"Features after downsampling: {len(tokenized_train)} (from {n_before})")

# model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

# training_args = TrainingArguments(
#     output_dir=OUTPUT_DIR,
#     per_device_train_batch_size=16,
#     num_train_epochs=NUM_EPOCHS,
#     learning_rate=3e-5,
#     fp16=True,
#     logging_steps=50,
#     save_strategy="steps",
#     save_steps=1000,
#     save_total_limit=3,       # keep only the 3 most recent checkpoints (disk space)
#     report_to="none",
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_train,
#     data_collator=default_data_collator,
# )

# # Resume from the last checkpoint if one exists (e.g. after an interruption)
# last_checkpoint = None
# if os.path.isdir(OUTPUT_DIR):
#     last_checkpoint = get_last_checkpoint(OUTPUT_DIR)
#     if last_checkpoint:
#         print(f"Resuming from checkpoint: {last_checkpoint}")

# print("Starting full-scale training run...")
# trainer.train(resume_from_checkpoint=last_checkpoint)

# trainer.save_model(OUTPUT_DIR)
# tokenizer.save_pretrained(OUTPUT_DIR)
# print(f"\nFull-scale fine-tuning complete. Model saved to: {OUTPUT_DIR}")
# print("Next step: proper evaluation with null-score-calibrated postprocessing "
#       "(not naive max-confidence) to get a real F1/EM number to report against "
#       "the Month 1 zero-shot baselines.")

#### Taking much more time than traing only  epoch 6 hours of compute time

In [6]:
"""
Month 2 - Lightweight Alternative: Token Classification (NER Formulation)
"""

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

# ---------------------------------------------------------------------------
# Step 0: Environment Setup
# ---------------------------------------------------------------------------
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU T4 x2 in Kaggle settings.")

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
MAX_LENGTH = 384
DOC_STRIDE = 128
OUTPUT_DIR = "/kaggle/working/legal-bert-ner-govlaw"

# ---------------------------------------------------------------------------
# Step 1: Load and Pivot the Data (Targeting a Single Clause)
# ---------------------------------------------------------------------------
print("Loading CUAD dataset...")
dataset = load_dataset("theatticusproject/cuad-qa", trust_remote_code=True)
df_train = pd.DataFrame(dataset["train"])

# Robust substring match instead of exact string match
df_target = df_train[df_train["question"].str.contains("govern", case=False, na=False)].reset_index(drop=True)

print(f"Isolated {len(df_target)} contracts for Governing Law.")

# Safety Check
if len(df_target) == 0:
    raise ValueError("Failed to isolate the clause. The filter found 0 matching questions.")
else:
    print(f"Sample Question Found: '{df_target['question'].iloc[0]}'")

# Convert back to HuggingFace Dataset
target_dataset = Dataset.from_pandas(df_target)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ---------------------------------------------------------------------------
# Step 2: Tokenization & Alignment (The "Highlighter" Logic)
# ---------------------------------------------------------------------------
def align_labels_with_tokens(examples):
    tokenized = tokenizer(
        examples["context"],
        truncation=True,
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")
    
    labels_list = []
    
    for i, offsets in enumerate(offset_mapping):
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]
        
        answer_spans = []
        for start_char, text in zip(answers["answer_start"], answers["text"]):
            answer_spans.append((start_char, start_char + len(text)))
            
        chunk_labels = []
        for token_start, token_end in offsets:
            if token_start == 0 and token_end == 0:
                chunk_labels.append(-100) # Ignore special tokens
                continue
                
            is_inside_answer = False
            for ans_start, ans_end in answer_spans:
                if token_start >= ans_start and token_end <= ans_end:
                    is_inside_answer = True
                    break
                    
            if is_inside_answer:
                chunk_labels.append(1) # Target Clause
            else:
                chunk_labels.append(0) # Background text
                
        labels_list.append(chunk_labels)
        
    tokenized["labels"] = labels_list
    return tokenized

print("Tokenizing, windowing, and aligning BIO labels...")
tokenized_dataset = target_dataset.map(
    align_labels_with_tokens,
    batched=True,
    remove_columns=target_dataset.column_names,
)

# ---------------------------------------------------------------------------
# Step 3: Hard Negative Mining (Compute Reduction)
# ---------------------------------------------------------------------------
def filter_hard_negatives(dataset, keep_empty_ratio=0.10):
    import random
    random.seed(42)
    
    keep_indices = []
    for i in range(len(dataset)):
        labels = dataset[i]["labels"]
        if 1 in labels:
            keep_indices.append(i)
        elif random.random() < keep_empty_ratio:
            keep_indices.append(i)
            
    return dataset.select(keep_indices)

print(f"Total chunks before filtering: {len(tokenized_dataset)}")
filtered_dataset = filter_hard_negatives(tokenized_dataset, keep_empty_ratio=0.10)
print(f"Total chunks after filtering: {len(filtered_dataset)}")

# ---------------------------------------------------------------------------
# Step 4: Model & Trainer Initialization
# ---------------------------------------------------------------------------
print(f"Loading Token Classification Model: {MODEL_NAME}")
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,
    num_train_epochs=2,
    learning_rate=3e-5,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=filtered_dataset,
    data_collator=data_collator,
)

# ---------------------------------------------------------------------------
# Step 5: Execute Training
# ---------------------------------------------------------------------------
print("Starting lightweight NER fine-tuning...")
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Training complete. Model saved to {OUTPUT_DIR}")

CUDA available: True
Loading CUAD dataset...
Isolated 428 contracts for Governing Law.
Sample Question Found: 'Highlight the parts (if any) of this contract related to "Governing Law" that should be reviewed by a lawyer. Details: Which state/country's law governs the interpretation of the contract?'
Tokenizing, windowing, and aligning BIO labels...


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

Total chunks before filtering: 19202
Total chunks after filtering: 2461
Loading Token Classification Model: nlpaueb/legal-bert-base-uncased


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored w

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete. Model saved to /kaggle/working/legal-bert-ner-govlaw


By reframing the task to token classification and applying hard negative filtering, the dataset was efficiently compressed from 19,202 chunks down to just 2,461 high-signal training windows. This optimization allowed the dual-epoch training to finish in just three minutes and eight seconds, completely eliminating the previous six-hour bottleneck. The steady convergence of the training loss from 0.285 down to 0.013 confirms the model successfully learned the specific lexical boundaries of the Governing Law clause without plateauing on empty background text.

Next Move: Inference and Extraction
You need to verify the model actually works on raw text. Run this inference script in a new cell to pass an unseen contract snippet through your newly saved model and extract the predicted words.

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Load the fine-tuned model and tokenizer from your output directory
MODEL_DIR = "/kaggle/working/legal-bert-ner-govlaw"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)

# A raw, unseen contract snippet containing a governing law clause mixed with background text
sample_text = (
    "The parties agree to keep all confidential information secure. "
    "This Agreement shall be governed by and construed in accordance with the laws of the State of California, "
    "without regard to its conflict of law principles. "
    "Both parties must sign the NDA before proceeding."
)

# Tokenize the input text
inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=384)

# Run inference (no gradients needed)
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Get the predicted class (0 or 1) for each token
predictions = torch.argmax(logits, dim=2).squeeze().tolist()
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze().tolist())

# Reconstruct the extracted text based on tokens predicted as '1' (Inside target clause)
extracted_tokens = []
for token, pred in zip(tokens, predictions):
    if pred == 1:
        # Remove subword hashes for clean reading
        clean_token = token.replace("##", "")
        extracted_tokens.append(clean_token)

# Rejoin the extracted tokens into a readable string
extracted_clause = " ".join(extracted_tokens)

print("--- RAW TEXT ---")
print(sample_text)
print("\n--- EXTRACTED GOVERNING LAW ---")
print(extracted_clause if extracted_clause else "No clause detected.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- RAW TEXT ---
The parties agree to keep all confidential information secure. This Agreement shall be governed by and construed in accordance with the laws of the State of California, without regard to its conflict of law principles. Both parties must sign the NDA before proceeding.

--- EXTRACTED GOVERNING LAW ---
this agreement shall be governed by and construed in accordance with the laws of the state of california , without regard to its conflict of law principles .


In [8]:
"""
Month 2 - Phase 3: Playbook Deviation & Risk Flagging
Purpose: Evaluate the extracted clause against standard corporate protocol 
using dense semantic embeddings and cosine similarity thresholds.
"""

# Install sentence-transformers if not already in your Kaggle environment
# !pip install -q sentence-transformers

import numpy as np
from sentence_transformers import SentenceTransformer, util

# 1. Load a lightweight, highly efficient semantic encoder
print("Loading semantic encoder...")
encoder = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Define the Institutional Playbook Standard
# Example: The company mandates New York jurisdiction.
playbook_standard = "This Agreement shall be governed by and construed in accordance with the laws of the State of New York."

# 3. The extracted clause from our NER model
extracted_clause = "this agreement shall be governed by and construed in accordance with the laws of the state of california , without regard to its conflict of law principles"

# 4. Generate Semantic Embeddings
playbook_embedding = encoder.encode(playbook_standard, convert_to_tensor=True)
extracted_embedding = encoder.encode(extracted_clause, convert_to_tensor=True)

# 5. Compute Cosine Similarity
similarity_score = util.cos_sim(playbook_embedding, extracted_embedding).item()

# 6. Flagging Logic
print("\n--- RISK EVALUATION REPORT ---")
print(f"Playbook Standard : {playbook_standard}")
print(f"Extracted Clause  : {extracted_clause}")
print(f"Similarity Score  : {similarity_score:.4f}")

if similarity_score >= 0.85:
    print("Risk Level        : [GREEN] Standard Protocol Compliant.")
elif similarity_score >= 0.50:
    print("Risk Level        : [YELLOW] Deviation Detected. Non-standard jurisdiction or terms.")
    # The system correctly flags Yellow because "California" + "conflict of law principles" 
    # deviates semantically from the strict "New York" playbook.
else:
    print("Risk Level        : [RED] Critical Deviation or Missing Clause.")

Loading semantic encoder...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


--- RISK EVALUATION REPORT ---
Playbook Standard : This Agreement shall be governed by and construed in accordance with the laws of the State of New York.
Extracted Clause  : this agreement shall be governed by and construed in accordance with the laws of the state of california , without regard to its conflict of law principles
Similarity Score  : 0.5938
Risk Level        : [YELLOW] Deviation Detected. Non-standard jurisdiction or terms.


Testing on a hardcoded, 50-word string proves the model works in theory, but real commercial contracts are 20-page PDFs.

In [9]:
!pip install -q PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 70.8 MB/s eta 0:00:00:00:0100:01


In [10]:

"""
End-to-End Pipeline: PDF Ingestion -> Sliding Window NER -> Risk Evaluation
"""

import pymupdf as fitz # PyMuPDF
import re
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from sentence_transformers import SentenceTransformer, util

# ---------------------------------------------------------
# 1. Configuration & Model Loading
# ---------------------------------------------------------
# Update this to point to a PDF you upload to Kaggle
PDF_PATH = "/kaggle/input/datasets/sumitnayek/sample-contract/Contract document.pdf" 
MODEL_DIR = "/kaggle/working/legal-bert-ner-govlaw"

print("Loading models (NER + Semantic Encoder)...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
ner_model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).to(device)
encoder = SentenceTransformer('all-MiniLM-L6-v2')

playbook_standard = "This Agreement shall be governed by and construed in accordance with the laws of the State of New York."
playbook_embedding = encoder.encode(playbook_standard, convert_to_tensor=True)

# ---------------------------------------------------------
# 2. PDF Parsing Utility (From Month 1)
# ---------------------------------------------------------
def parse_contract_pdf(file_path):
    print(f"Parsing PDF: {file_path}")
    doc = fitz.open(file_path)
    full_text = []
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        blocks = page.get_text("blocks")
        for block in blocks:
            if block[6] == 0:  # Text block
                text = block[4].strip()
                text = re.sub(r'\n+', ' ', text)
                if len(text) > 5:
                    full_text.append(text)
    return "\n\n".join(full_text)

# ---------------------------------------------------------
# 3. Sliding-Window Inference & Extraction
# ---------------------------------------------------------
def extract_clause_from_document(full_text):
    print("Running sliding-window inference across full document...")
    # Tokenize the entire document into overlapping 384-token windows
    inputs = tokenizer(
        full_text, 
        truncation=True, 
        max_length=384, 
        stride=128, 
        return_overflowing_tokens=True, 
        return_offsets_mapping=True, 
        padding="max_length", 
        return_tensors="pt"
    )
    
    offset_mapping = inputs.pop("offset_mapping").numpy()
    inputs.pop("overflow_to_sample_mapping")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Run inference on all chunks
    with torch.no_grad():
        logits = ner_model(**inputs).logits
        preds = torch.argmax(logits, dim=2).cpu().numpy()
        
    # Deduplicate extracted characters using a Set
    # (If a clause falls in the overlap of two chunks, we don't want it printed twice)
    char_indices = set()
    
    for chunk_idx in range(len(preds)):
        for seq_idx in range(len(preds[chunk_idx])):
            if preds[chunk_idx][seq_idx] == 1: # Predicted as Target Clause
                start, end = offset_mapping[chunk_idx][seq_idx]
                if start != 0 or end != 0: # Ignore [CLS], [SEP]
                    for char_idx in range(start, end):
                        char_indices.add(char_idx)
                        
    if not char_indices:
        return None
        
    # Reconstruct the exact original string from the text
    # Reconstruct the exact original string from the text
    sorted_indices = sorted(list(char_indices))
    extracted_str = ""
    last_idx = sorted_indices[0] - 1
    
    for idx in sorted_indices:
        if idx > last_idx + 1:
            # Check what characters were skipped
            gap_text = full_text[last_idx + 1 : idx]
            if gap_text.isspace():
                extracted_str += gap_text  # It was just a space/newline, keep it
            else:
                extracted_str += " ... "   # It was a real gap between different clauses
                
        extracted_str += full_text[idx]
        last_idx = idx
        
    return extracted_str.strip()

# ---------------------------------------------------------
# 4. Execution & Risk Flagging
# ---------------------------------------------------------
try:
    # 1. Parse Document
    contract_text = parse_contract_pdf(PDF_PATH)
    
    # 2. Extract Clause
    extracted_clause = extract_clause_from_document(contract_text)
    
    print("\n" + "="*50)
    print("RESULTS")
    print("="*50)
    
    if extracted_clause:
        print(f"EXTRACTED CLAUSE:\n{extracted_clause}\n")
        
        # 3. Evaluate Risk
        extracted_embedding = encoder.encode(extracted_clause, convert_to_tensor=True)
        similarity = util.cos_sim(playbook_embedding, extracted_embedding).item()
        
        print(f"PLAYBOOK STANDARD: {playbook_standard}")
        print(f"SIMILARITY SCORE:  {similarity:.4f}")
        
        if similarity >= 0.85:
            print("RISK FLAG: [GREEN] Compliant")
        elif similarity >= 0.50:
            print("RISK FLAG: [YELLOW] Deviation (Non-standard terms detected)")
        else:
            print("RISK FLAG: [RED] Critical Deviation")
    else:
        print("RISK FLAG: [RED] Clause not found in document.")

except Exception as e:
    print(f"Execution Error: {e}")
    print("Ensure you have uploaded a PDF and set the correct PDF_PATH.")

Loading models (NER + Semantic Encoder)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Parsing PDF: /kaggle/input/datasets/sumitnayek/sample-contract/Contract document.pdf
Running sliding-window inference across full document...

RESULTS
EXTRACTED CLAUSE:
The contract shall be governed by and constructed according to the laws in

force in India.

PLAYBOOK STANDARD: This Agreement shall be governed by and construed in accordance with the laws of the State of New York.
SIMILARITY SCORE:  0.4830
RISK FLAG: [RED] Critical Deviation


In [11]:
"""
Model Evaluation & Benchmarking Suite
Compares Fine-Tuned Legal-BERT NER against Zero-Shot QA Baseline (DistilBERT-SQuAD)
across standard CUAD evaluation parameters: EM, Precision, Recall, F1, and Latency.
"""

import time
import string
import re
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    pipeline
)

# ---------------------------------------------------------------------------
# 1. Metric Helper Functions (Standard SQuAD/CUAD Normalization)
# ---------------------------------------------------------------------------
def normalize_text(text):
    """Lower text and remove punctuation, articles and extra whitespace."""
    def remove_articles(t):
        return re.sub(r'\b(a|an|the)\b', ' ', t)
    def white_space_fix(t):
        return ' '.join(t.split())
    def remove_punc(t):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in t if ch not in exclude)
    return white_space_fix(remove_articles(remove_punc(text.lower())))

def compute_exact_match(prediction, ground_truth):
    return int(normalize_text(prediction) == normalize_text(ground_truth))

def compute_f1(prediction, ground_truth):
    pred_tokens = normalize_text(prediction).split()
    gt_tokens = normalize_text(ground_truth).split()
    
    if len(pred_tokens) == 0 and len(gt_tokens) == 0:
        return 1.0, 1.0, 1.0  # Both empty -> correct null prediction
    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return 0.0, 0.0, 0.0

    common_tokens = set(pred_tokens) & set(gt_tokens)
    if len(common_tokens) == 0:
        return 0.0, 0.0, 0.0

    precision = len(common_tokens) / len(pred_tokens)
    recall = len(common_tokens) / len(gt_tokens)
    f1 = 2 * (precision * recall) / (precision + recall)
    return precision, recall, f1

# ---------------------------------------------------------------------------
# 2. Load Models
# ---------------------------------------------------------------------------
device = 0 if torch.cuda.is_available() else -1
NER_MODEL_DIR = "/kaggle/working/legal-bert-ner-govlaw"

print("Loading Fine-Tuned Legal-BERT NER model...")
ner_tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_DIR)
ner_model = AutoModelForTokenClassification.from_pretrained(NER_MODEL_DIR)
if torch.cuda.is_available():
    ner_model = ner_model.cuda()

print("Loading Benchmark Baseline (Zero-Shot DistilBERT QA)...")
baseline_qa = pipeline(
    "question-answering", 
    model="distilbert-base-cased-distilled-squad", 
    device=device
)

# ---------------------------------------------------------------------------
# 3. Load CUAD Test Split
# ---------------------------------------------------------------------------
print("Loading CUAD test dataset...")
dataset = load_dataset("theatticusproject/cuad-qa", trust_remote_code=True)
df_test = pd.DataFrame(dataset["test"])

# Filter for Governing Law questions in test set
test_samples = df_test[df_test["question"].str.contains("govern", case=False, na=False)].reset_index(drop=True)

# Select evaluation sample size (e.g., 50 contracts)
EVAL_N = min(50, len(test_samples))
test_subset = test_samples.head(EVAL_N)
print(f"Benchmarking on {EVAL_N} unseen test contracts.\n")

# ---------------------------------------------------------------------------
# 4. Evaluation Loop
# ---------------------------------------------------------------------------
ner_metrics = {"em": [], "precision": [], "recall": [], "f1": [], "latency": []}
base_metrics = {"em": [], "precision": [], "recall": [], "f1": [], "latency": []}

QUESTION_PROMPT = "Which state or country's law governs the interpretation of the contract?"

for idx, row in test_subset.iterrows():
    context = row["context"]
    ground_truth = row["answers"]["text"][0] if len(row["answers"]["text"]) > 0 else ""

    # --- Benchmark: DistilBERT Zero-Shot QA ---
    t0 = time.perf_counter()
    try:
        # Standard pipelines truncate long inputs
        qa_input = context[:2000]
        base_res = baseline_qa(question=QUESTION_PROMPT, context=qa_input)
        base_pred = base_res["answer"]
    except Exception:
        base_pred = ""
    t_base = (time.perf_counter() - t0) * 1000

    p_base, r_base, f1_base = compute_f1(base_pred, ground_truth)
    em_base = compute_exact_match(base_pred, ground_truth)
    
    base_metrics["em"].append(em_base)
    base_metrics["precision"].append(p_base)
    base_metrics["recall"].append(r_base)
    base_metrics["f1"].append(f1_base)
    base_metrics["latency"].append(t_base)

    # --- Candidate: Fine-Tuned Legal-BERT NER ---
    t0 = time.perf_counter()
    inputs = ner_tokenizer(
        context, truncation=True, max_length=384, stride=128, 
        return_overflowing_tokens=True, return_offsets_mapping=True, 
        padding="max_length", return_tensors="pt"
    )
    offset_mapping = inputs.pop("offset_mapping").numpy()
    inputs.pop("overflow_to_sample_mapping")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    
    with torch.no_grad():
        logits = ner_model(**inputs).logits
        preds = torch.argmax(logits, dim=2).cpu().numpy()

    char_indices = set()
    for chunk_idx in range(len(preds)):
        for seq_idx in range(len(preds[chunk_idx])):
            if preds[chunk_idx][seq_idx] == 1:
                start, end = offset_mapping[chunk_idx][seq_idx]
                if start != 0 or end != 0:
                    for c_idx in range(start, end):
                        char_indices.add(c_idx)

    if char_indices:
        sorted_indices = sorted(list(char_indices))
        extracted_chars = []
        last_idx = sorted_indices[0] - 1
        for c in sorted_indices:
            if c > last_idx + 1 and not context[last_idx + 1:c].isspace():
                extracted_chars.append(" ")
            extracted_chars.append(context[c])
            last_idx = c
        ner_pred = "".join(extracted_chars).strip()
    else:
        ner_pred = ""
    t_ner = (time.perf_counter() - t0) * 1000

    p_ner, r_ner, f1_ner = compute_f1(ner_pred, ground_truth)
    em_ner = compute_exact_match(ner_pred, ground_truth)

    ner_metrics["em"].append(em_ner)
    ner_metrics["precision"].append(p_ner)
    ner_metrics["recall"].append(r_ner)
    ner_metrics["f1"].append(f1_ner)
    ner_metrics["latency"].append(t_ner)

# ---------------------------------------------------------------------------
# 5. Output Summary Table
# ---------------------------------------------------------------------------
results = pd.DataFrame({
    "Evaluation Parameter": [
        "Exact Match (EM %)", 
        "Token Precision (%)", 
        "Token Recall (%)", 
        "Macro F1-Score (%)", 
        "Avg Latency (ms/doc)"
    ],
    "Benchmark (DistilBERT QA)": [
        f"{np.mean(base_metrics['em']) * 100:.2f}%",
        f"{np.mean(base_metrics['precision']) * 100:.2f}%",
        f"{np.mean(base_metrics['recall']) * 100:.2f}%",
        f"{np.mean(base_metrics['f1']) * 100:.2f}%",
        f"{np.mean(base_metrics['latency']):.1f} ms"
    ],
    "Fine-Tuned Legal-BERT NER": [
        f"{np.mean(ner_metrics['em']) * 100:.2f}%",
        f"{np.mean(ner_metrics['precision']) * 100:.2f}%",
        f"{np.mean(ner_metrics['recall']) * 100:.2f}%",
        f"{np.mean(ner_metrics['f1']) * 100:.2f}%",
        f"{np.mean(ner_metrics['latency']):.1f} ms"
    ]
})

print("=" * 65)
print("BENCHMARK EVALUATION RESULTS (CUAD TEST SET)")
print("=" * 65)
print(results.to_markdown(index=False))

Loading Fine-Tuned Legal-BERT NER model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading Benchmark Baseline (Zero-Shot DistilBERT QA)...


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading CUAD test dataset...
Benchmarking on 50 unseen test contracts.



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


BENCHMARK EVALUATION RESULTS (CUAD TEST SET)
| Evaluation Parameter   | Benchmark (DistilBERT QA)   | Fine-Tuned Legal-BERT NER   |
|:-----------------------|:----------------------------|:----------------------------|
| Exact Match (EM %)     | 2.00%                       | 12.00%                      |
| Token Precision (%)    | 21.50%                      | 12.67%                      |
| Token Recall (%)       | 2.90%                       | 12.02%                      |
| Macro F1-Score (%)     | 3.66%                       | 12.04%                      |
| Avg Latency (ms/doc)   | 36.1 ms                     | 1068.2 ms                   |
